# Block 4 — Stateful streaming: event-time windows, watermarks, state recovery

Reads `landing/events/` and maintains a 5-minute tumbling revenue aggregate on **event time**,
into `retail.gold.revenue_5min`.

Proves four things:
1. Output is keyed by **when the event happened**, not when the file arrived.
2. A late row **inside** the watermark is counted into an already-populated window.
3. A late row **outside** the watermark is **dropped**, and the engine reports the count.
4. Window state survives a full query restart — a window populated in one run gains rows in the next.

**Scope, stated honestly:** there is no broker and no live producer. Event time comes from a column
in a CSV. What is real here is the *stateful* machinery — watermark, state store, append-mode
emission — not the transport.

In [0]:
STG     = "stgaccdeprep"
LANDING = f"abfss://landing@{STG}.dfs.core.windows.net"

SRC       = f"{LANDING}/events"
CHKPT_LOC = f"{LANDING}/_control/events_windowed/checkpoint"
TARGET    = "retail.gold.revenue_5min"

WINDOW    = "5 minutes"
WATERMARK = "10 minutes"

from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               TimestampType, DecimalType)

display(dbutils.fs.ls(SRC))

### Look at the bytes before typing them

Block 3's lesson, applied without being asked. Excel renders `2026-08-10 10:00:04` as
`10-08-2026 10:00`; a schema written against that rendering produces **null timestamps with no
error at all** — and a null event time means every row is silently late.

Read the text first, then declare the schema against what is actually in the file.

In [0]:
display(spark.read.text(f"{SRC}/events_01_late.csv").limit(4))

### Declared schema here, hints in Block 3 — the same decision, decided the other way

Block 3 used `schemaHints` because the whole point was to watch a column arrive. Here the contract
is fixed and I own it, so it is **declared**: a missing or retyped column fails at read time
instead of somewhere downstream.

Declaring the schema also sets `schemaEvolutionMode` to `none`, which is exactly right — and it
means this stream needs only **one** control location (the checkpoint), where Block 3 needed two.

*Trade-off: I chose a declared schema over hints because it optimised for a loud failure on a
contract I control, at the cost of the stream being unable to absorb a new column. In Block 3's
context — an upstream I do not control, where a new column is expected — hints were right.*

`amount` is `DECIMAL(12,2)`, carried from Block 1 for the same reason: the windowed sums have to
reconcile exactly, not within tolerance.

In [0]:
events_schema = StructType([
    StructField("event_id",    StringType(),       True),
    StructField("customer_id", StringType(),       True),
    StructField("event_ts",    TimestampType(),    True),
    StructField("amount",      DecimalType(12, 2), True),
    StructField("event_type",  StringType(),       True),
])

raw = (spark.readStream.format("cloudFiles")
       .option("cloudFiles.format", "csv")
       .option("cloudFiles.maxFilesPerTrigger", 1)
       .option("header", "true")
       .schema(events_schema)
       .load(SRC))

raw.printSchema()

### One file per micro-batch — the option this block does not work without

`cloudFiles.maxFilesPerTrigger = 1`.

The watermark is derived from the maximum event time seen in **previous** micro-batches. With all
three files consumed in a single batch, every row — the late ones included — would be evaluated
against a watermark of zero, nothing would ever be dropped, and the demonstration would silently
not happen while appearing to succeed.

Forcing one file per batch makes the watermark advance in observable steps. **The instrument has
to be slower than the thing it measures.**

### The watermark is a bound on lateness, not a filter on rows

`withWatermark("event_ts", "10 minutes")` is a promise to the engine: *no row will arrive more than
10 minutes (in event time) behind the latest event time I have already seen.* The engine uses that
promise to decide two things it otherwise could not: when a window is final and can be emitted, and
when its state can be evicted. Without it, append-mode windowed state grows forever.

**The two-watermarks trap.** An ADF incremental-load watermark is a high-water mark in a control
table, used to decide *what to extract next*. A streaming event-time watermark is a lateness bound
used to decide *when state can be evicted*. Same word, unrelated mechanics — and they sit one
sentence apart on this resume.

The two `late_*_rows` columns below are a **test instrument, not a business measure** — they count the
planted late rows so the proof is a table read rather than an inference.

**A known flaw in the synthetic data:** `event_type` is independent of sign, so refunds *increase*
`revenue`. The column is really gross amount. A correct model signs them:
`F.sum(F.when(F.col("event_type") == "REFUND", -F.col("amount")).otherwise(F.col("amount")))`.
Left as-is because the counts are what this block proves, but it would be wrong in production.

In [0]:
LATE_INSIDE  = F.lit("4444.44").cast("decimal(12,2)")   # late, but within the 10-min bound
LATE_OUTSIDE = F.lit("9999.99").cast("decimal(12,2)")   # late, and far outside it

windowed = (raw
    .withWatermark("event_ts", WATERMARK)
    .groupBy(F.window("event_ts", WINDOW))
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount").alias("revenue"),
        F.approx_count_distinct("customer_id").alias("distinct_customers"),
        F.sum((F.col("amount") == LATE_INSIDE ).cast("int")).alias("late_inside_rows"),
        F.sum((F.col("amount") == LATE_OUTSIDE).cast("int")).alias("late_outside_rows"),
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "event_count", "revenue", "distinct_customers",
        "late_inside_rows", "late_outside_rows",
    ))

windowed.printSchema()

### Append mode, and why the last window is always missing

`outputMode("append")` emits a window **once, when it is final** — that is, once the watermark has
passed the window's end. It is the only mode that produces an immutable, appendable gold table.

The consequence is unintuitive and it is the hour the runbook warned about: **the most recent
window never appears.** Nothing is broken. There is simply no later data yet to push the watermark
past its end. `update` mode would show it immediately and re-emit it on every change, which is
useful for a live dashboard and wrong for a table other jobs read.

*Trade-off: append over update, because gold is read by downstream jobs that must not see a row
change under them, at the cost of latency equal to the watermark.*

In [0]:
q = (windowed.writeStream
     .format("delta")
     .outputMode("append")
     .option("checkpointLocation", CHKPT_LOC)
     .trigger(availableNow=True)
     .toTable(TARGET))

q.awaitTermination()

for p in q.recentProgress:
    so = (p.get("stateOperators") or [{}])[0]
    print(f"batch {p['batchId']}: "
          f"in={p['numInputRows']:>4}  "
          f"watermark={p.get('eventTime', {}).get('watermark')}  "
          f"dropped_by_watermark={so.get('numRowsDroppedByWatermark')}  "
          f"state_rows={so.get('numRowsTotal')}")

### The emitted windows

`event_count` is the census. `late_outside_rows` must be **0 in every row** — those three rows are
not supposed to be anywhere in this table.

In [0]:
print("windows emitted :", spark.table(TARGET).count())
print("rows accounted  :", spark.table(TARGET).agg(F.sum("event_count")).collect()[0][0])
display(spark.table(TARGET).orderBy("window_start"))

## Run 1 — `events_01_late.csv`

120 rows, 10:00:00–10:29:00, spread evenly over six 5-minute windows.

Expected: `batch 0: in=120`, `dropped_by_watermark=0`, watermark advances to **10:19:00**
(10:29:00 − 10 minutes). Windows ending at or before 10:19 are final and emit; `[10:15,10:20)`
ends at 10:20 and is held.

**Result:**
`batch 0: in=120  watermark=1970-01-01T00:00:00.000Z  dropped=0  state_rows=6`
`batch 1: in=  0  watermark=2026-08-10T10:19:00.000Z  dropped=0  state_rows=3`

Batch 0 carries **no watermark at all** — literally epoch. Nothing has been seen, so nothing
can be late, and all six windows build. Nothing is emitted.

The output came from **batch 1, which read zero rows.** The watermark computed at the end of a
batch is not applied until the next one, and `availableNow` runs a final empty batch to flush.
State fell 6 → 3 as three finished windows were emitted and evicted.

Block 3 also logged a `0 rows` batch and it did nothing at all. Here an empty batch advances the
watermark, publishes three windows and frees their state. **In a stateful stream, time is an input.**

## Run 2 — `events_02_late.csv` arrives

90 rows, 10:30:00–10:44:00, three windows of 30. All are ahead of the 10:19 watermark, so nothing
is dropped. Watermark advances to **10:34:00**.

**The first half of the recovery proof:** the six windows from run 1 are still in the table, and
`[10:30,10:35)` is now accumulating in a state store that a *different query instance* created.

**Result:**
`batch 2: in= 90  watermark=2026-08-10T10:19:00.000Z  dropped=0  state_rows=6`
`batch 3: in=  0  watermark=2026-08-10T10:34:00.000Z  dropped=0  state_rows=3`

Six windows, 120 rows. The three rows emitted in run 1 are byte-identical — append mode publishes
once and never revises.

`state_rows` counts **windows, not events**: three state rows holding 90 events. State size tracks
the number of open windows, not data volume.

## Run 3 — `events_03_late.csv`, the one that matters

30 rows, and six of them are planted. The watermark in force during this batch is **10:34:00**.

| Rows | `event_ts` | vs watermark | Expected |
|---|---|---|---|
| 3 × `4444.44` | 10:36:00 | **after** 10:34 | **counted** into `[10:35,10:40)`, which has not emitted yet |
| 3 × `9999.99` | 10:05:00 | **far before** 10:34 | **dropped** — its window closed two runs ago |
| 24 on-time | 10:45–10:59 | ahead | counted |

Both marker sets are *late* — every one of them is older than the 10:44 already seen. The watermark
is the only thing separating them, and it separates them by 31 minutes of event time.

Expected: `dropped_by_watermark=3`, and `[10:35,10:40)` closes at **33** rows, not 30 — the three
extra arriving in a **later query instance** than the 30 they join.

**Result:**
`batch 4: in= 30  watermark=2026-08-10T10:34:00.000Z  dropped_by_watermark=1  state_rows=6`
`batch 5: in=  0  watermark=2026-08-10T10:49:00.000Z  dropped_by_watermark=0  state_rows=3`

`[10:35,10:40)` closed at **33**, `late_inside_rows = 3`. No window anywhere has
`late_outside_rows > 0`. Nine windows, 213 rows emitted; 24 rows still held in three open windows.

**The counter reported 1, not 3 — and it is not wrong.** All three late rows share the window key
`10:05–10:10`, so the partial aggregation ahead of the state operator combines them into a single
row, and that one row is what the watermark rejects. `numRowsDroppedByWatermark` counts rows
arriving at the state operator, **not raw input records**.

Counted from the data instead: 240 landed − 213 emitted − 24 still open = **3 records lost**. And
the window itself: **23 rows in the source file, 20 in the published table.**

So the metric is a correct alarm and a wrong tally. Alert on `> 0`; never report it as a record count.

### The proof, as one table read

In [0]:
t = spark.table(TARGET)

print("late rows accepted (expect 3 in [10:35,10:40)):")
display(t.filter("late_inside_rows > 0")
         .select("window_start", "window_end", "event_count", "late_inside_rows", "revenue"))

print("late rows dropped — this must return zero rows:")
display(t.filter("late_outside_rows > 0"))

print("the window that gained rows across a restart:")
display(t.filter("window_start = '2026-08-10 10:35:00'"))

print()
print("windows emitted   :", t.count())
print("rows in windows   :", t.agg(F.sum("event_count")).collect()[0][0])
print("late kept         :", t.agg(F.sum("late_inside_rows")).collect()[0][0],  "(expect 3)")
print("late present      :", t.agg(F.sum("late_outside_rows")).collect()[0][0], "(expect 0)")

### Observed output

| window_start | window_end | event_count | late_inside_rows | revenue |
|---|---|---|---|---|
| 10:35:00 | 10:40:00 | **33** | **3** | 177749.20 |

One row in the entire table has `late_inside_rows > 0`, and it is the window that reads 33
instead of 30. The `late_outside_rows > 0` filter returns **no rows**.

The nine emitted windows: 20, 20, 20, 20, 20, 20, 30, **33**, 30 — a 20 came from file 1,
a 30 from file 2, and the 33 is 30 plus three late arrivals from file 3.

*(Tables render via `display()`, which does not serialize into the exported notebook.
Screenshots are in the repo.)*

### Reconciliation

240 rows landed across three files. 3 were dropped by the watermark, so 237 were accepted. Of
those, the emitted windows account for some and the rest are still held in the state store waiting
for a watermark that will never come — because no further data will arrive.

`emitted + still_in_state = 237` is the check. **Rows that vanish from both sides of that equation
are the failure mode this whole block exists to make visible.**

In [0]:
src = spark.read.option("header", True).schema(events_schema).csv(SRC)
gold = spark.table(TARGET)

src_w = src.withColumn("window_start", F.window("event_ts", WINDOW).getField("start"))

landed     = src.count()
emitted    = gold.agg(F.sum("event_count")).collect()[0][0]
still_open = src_w.join(gold.select("window_start").distinct(),
                        on="window_start", how="left_anti").count()

print(f"landed in files   : {landed}")
print(f"emitted into gold : {emitted}")
print(f"still in state    : {still_open}   (rows whose window was never emitted)")
print(f"unaccounted for   : {landed - emitted - still_open}   <-- dropped by the watermark")
print()
print("The counter said dropped_by_watermark = 1, because three rows sharing one")
print("window key are pre-aggregated into a single row before the state operator.")
print("Counted from the data instead, the true number of lost records is above.")
print()

WIN = "2026-08-10 10:05:00"
print(f"source rows in window {WIN} :",
      src_w.filter(F.col("window_start") == WIN).count())
print(f"gold event_count for it        :",
      gold.filter(F.col("window_start") == WIN).collect()[0]["event_count"])

display(src_w.join(gold.select("window_start").distinct(),
                   on="window_start", how="left_anti")
             .groupBy("window_start").count().orderBy("window_start"))

### `batchWatermarkMs` — Block 3's instrument, now non-zero

Block 3 read `0` here because there was no stateful operator. It is the watermark the engine
actually used, frozen into the offset log, and it is the fastest way to answer *"why did nothing
come out?"* — compare it to the window end you are waiting for and the answer is arithmetic.

The state store is `RocksDBStateStoreProvider`, also named in this file. That is what survives a
restart, and it is a different thing from the Delta transaction log: the log makes the **output**
exactly-once, the state store makes the **computation** resumable.

Two more things this file records. `spark.sql.shuffle.partitions: 200` — state is partitioned
across those 200 partitions, and the value is **frozen into the checkpoint**. Change it later and
the state partitioning is invalid; the setting has to be sized before the first run. For nine
windows it is absurd over-partitioning, which is exactly why it belongs in a tuning conversation.

And `batchTimestampMs` — the wall clock frozen at batch time, so a replayed batch reproduces its
own notion of "now". Same mechanism as Block 3's `_ingest_ts` on the replayed rows.

In [0]:
display(dbutils.fs.ls(f"{CHKPT_LOC}/offsets"))

for n in ["0", "1", "2"]:
    try:
        print(f"--- offsets/{n} ---")
        print(dbutils.fs.head(f"{CHKPT_LOC}/offsets/{n}", 900))
    except Exception as e:
        print("  not present:", type(e).__name__)

In [0]:
import json
from datetime import datetime, timezone

display(dbutils.fs.ls(f"{CHKPT_LOC}/offsets"))

for n in range(6):
    meta = json.loads(dbutils.fs.head(f"{CHKPT_LOC}/offsets/{n}", 2000).split("\n")[1])
    ms = meta.get("batchWatermarkMs")
    when = datetime.fromtimestamp(ms/1000, timezone.utc).strftime("%H:%M:%S") if ms else "epoch — no watermark yet"
    print(f"offsets/{n}   batchWatermarkMs = {ms:>14}   -> {when}")

### OPTIONAL — stream-static join

Run only if there is time. Enriches each event with its customer's **current** dimension row and
writes `retail.silver.events_enriched`.

The point worth stating: a stream-static join **re-reads the static side on every micro-batch**, so
a dimension update is picked up without restarting the stream. That is what makes it cheap — no
state is kept for the static side. A stream-*stream* join is a different animal: both sides need
watermarks, and the engine has to buffer.

Joining `is_current` only is a **Type 1 read of a Type 2 dimension** — the same call made in gold,
and wrong for a historical restatement, which would need `valid_from`/`valid_to` bracketing
`event_ts`.

In [0]:
ENRICHED  = "retail.silver.events_enriched"
CHKPT_ENR = f"{LANDING}/_control/events_enriched/checkpoint"

dim_current = (spark.table("retail.silver.dim_customer")
                    .filter("is_current")
                    .select("customer_id", "customer_name", "city", "segment"))

enriched = (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .schema(events_schema)
            .load(SRC)
            .join(F.broadcast(dim_current), on="customer_id", how="left"))

qe = (enriched.writeStream
      .format("delta")
      .outputMode("append")
      .option("checkpointLocation", CHKPT_ENR)
      .trigger(availableNow=True)
      .toTable(ENRICHED))
qe.awaitTermination()

e = spark.table(ENRICHED)
print("enriched rows :", e.count(), "(expect 240 — no watermark here, so nothing is dropped)")
print("unmatched     :", e.filter("customer_name IS NULL").count(), "(expect 0)")
display(e.groupBy("segment").agg(F.count("*").alias("events"),
                                 F.sum("amount").alias("revenue")).orderBy(F.desc("revenue")))

### Two streams, one input, three rows apart

| stream | watermark | rows landed |
|---|---|---|
| `events_enriched` | none | **240** |
| `revenue_5min` | 10 minutes | **237** (213 emitted + 24 still open) |

Same three files, same cluster, same run. The only difference between those numbers is that one
stream made a promise about lateness and the other did not. The watermark is the three rows.